# Persian Pragmatics Benchmark: Response Selection (NLI / Multiple Choice)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abebr/persian-pragmatics-dataset/blob/main/notebooks/baselines.ipynb)

## 🎯 Task Formulation
Given a grounded context and an indirect Persian utterance, the model must choose between:
- **Label 0 (`Correct Response`):** Socioculturally and pragmatically appropriate reaction.
- **Label 1 (`Naive LLM Response`):** Literal translation trap / surface hallucination.

This benchmark rigorously penalizes keyword heuristics (Random Guess = 50.0%).

In [ ]:
# 1. Install dependencies
!pip install -q transformers datasets evaluate accelerate scikit-learn pandas

In [ ]:
# 2. Load dataset directly from Hugging Face Hub
import random
import pandas as pd
from datasets import load_dataset

dataset = load_dataset('abebr/persian-pragmatics-dataset')
print('Dataset loaded successfully:')
print(dataset)

def build_pairwise_pairs(split_data, seed=42):
    rng = random.Random(seed)
    records = []
    for row in split_data:
        ctx_utt = f"{row['context']} گوینده: «{row['utterance']}»"
        correct = row['correct_response']
        naive = row['naive_llm_response']
        
        # 50% target 0 (A is correct), 50% target 1 (B is correct)
        if rng.random() < 0.5:
            opt_a, opt_b, target = correct, naive, 0
        else:
            opt_a, opt_b, target = naive, correct, 1
            
        records.append({
            'id': row['id'],
            'category': row['category'],
            'prompt': ctx_utt,
            'option_a': opt_a,
            'option_b': opt_b,
            'text': f"{ctx_utt} [گزینه الف: {opt_a}] [گزینه ب: {opt_b}]",
            'label': target
        })
    return pd.DataFrame(records)

train_df = build_pairwise_pairs(dataset['train'], seed=42)
test_df = build_pairwise_pairs(dataset['test'], seed=42)
print(f'Train samples: {len(train_df):,} | Test samples: {len(test_df):,}')
print('\nSample task input:\n', train_df['text'].iloc[0])

## 1. Classical Baseline: TF-IDF + Logistic Regression
Demonstrates that surface word overlap alone cannot solve pragmatic intent.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

vec = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_vec = vec.fit_transform(train_df['text'])
X_test_vec = vec.transform(test_df['text'])

clf = LogisticRegression(max_iter=1000, random_state=42)
clf.fit(X_train_vec, train_df['label'])

preds = clf.predict(X_test_vec)
tfidf_acc = accuracy_score(test_df['label'], preds)

print(f'=== TF-IDF + Logistic Regression ===')
print(f'Accuracy: {tfidf_acc*100:.2f}% (Expected ~50% random chance)')
print(classification_report(test_df['label'], preds, target_names=['Choice A', 'Choice B']))

## 2. Neural Baseline: ParsBERT (`HooshvareLab/bert-fa-base-uncased`)
Fine-tunes a Persian pre-trained Transformer to select the pragmatically coherent option.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import evaluate
import numpy as np

model_ckpt = 'HooshvareLab/bert-fa-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(batch):
    return tokenizer(
        batch['prompt'],
        [f"الف: {a} | ب: {b}" for a, b in zip(batch['option_a'], batch['option_b'])],
        truncation=True,
        max_length=160,
        padding='max_length'
    )

train_hf = Dataset.from_pandas(train_df[['prompt', 'option_a', 'option_b', 'label']])
test_hf = Dataset.from_pandas(test_df[['prompt', 'option_a', 'option_b', 'label']])

train_tok = train_hf.map(tokenize_fn, batched=True)
test_tok = test_hf.map(tokenize_fn, batched=True)

metric_acc = evaluate.load('accuracy')
metric_f1 = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=preds, references=labels)['accuracy']
    f1 = metric_f1.compute(predictions=preds, references=labels, average='macro')['f1']
    return {'accuracy': acc, 'macro_f1': f1}

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=2)

training_args = TrainingArguments(
    output_dir='./results_parsbert_pragmatics',
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    report_to='none',
    fp16=torch.cuda.is_available()
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=test_tok,
    compute_metrics=compute_metrics
)

print('Starting ParsBERT fine-tuning on Pragmatic Response Selection...')
trainer.train()

eval_res = trainer.evaluate()
print('\n=== ParsBERT Final Evaluation on 1,000 Test Set ===')
print(f'Accuracy: {eval_res["eval_accuracy"]*100:.2f}%')
print(f'Macro F1: {eval_res["eval_macro_f1"]*100:.2f}%')

## 3. Results Summary Table for Paper
Copy this Markdown table directly into your paper's baseline comparison section.

In [ ]:
results = [
    {'Model': 'Random Guess', 'Type': 'Theoretical Chance', 'Accuracy': '50.0%', 'Macro F1': '50.0%'},
    {'Model': 'TF-IDF + Logistic Regression', 'Type': 'Classical ML', 'Accuracy': f'{tfidf_acc*100:.1f}%', 'Macro F1': f'{tfidf_f1*100:.1f}%'},
    {'Model': 'ParsBERT (bert-fa-base-uncased)', 'Type': 'Persian Transformer PLM', 'Accuracy': f'{eval_res["eval_accuracy"]*100:.1f}%', 'Macro F1': f'{eval_res["eval_macro_f1"]*100:.1f}%'}
]

res_df = pd.DataFrame(results)
print(res_df.to_markdown(index=False))